<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week8_Lecture1_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

## Week 8: Lecture 1: Data Cleaning and Preparation
---

**Agenda**
- Missing values are None, numpy.nan
- DataFrame.isnull() detects missing values
- DataFrame.notnull() detects non-missing values
- DataFrame.dropna(axis=0, how='any', thresh=None, subset=None, inplace=False)[source]
- DataFrame.fillna(value=None, method=None, axis=None, inplace=False, limit=None, downcast=None)
- DataFrame.transform() performs operations element-wise
- duplicated() returns a boolean Series indicating whether each row is a duplicate
- drop_duplicates() returns a DataFrame where the duplicated array is removed:
- Transform data by applying function
- data.replace([-999, -1000], np.nan): replace values
- data.index.map(transform): rename index
- data.rename(index=str.title, columns=str.upper): rename index and columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Handling Missing Data

Missing data occurs commonly in many data analysis applications.
All of the descriptive statistics on pandas objects exclude missing data by default. For numeric data, pandas uses the floating-point value NaN (Not a Number) to represent missing data. We call this a sentinel value that can be easily detected.

## `isnull()`: detects missing values
```
string_data = pd.Series(['aardvark', 'artichoke', np.nan, 'avocado'])

string_data.isnull()

string_data.isnull().sum()
```

## Exercise:
Run and test the above code.

## Dealing with Missing Data
In statistics applications, NA data may either be data that does not exist or that exists but was not observed
(through problems with data collection, for example). When cleaning up data for
analysis, it is often important to do analysis on the missing data itself to identify data
collection problems or potential biases in the data caused by missing data.

  
## `dropna()`: Drops missing values; for Series, it returns non-null data.

```
from numpy import nan as NA
data = pd.Series([1, NA, 3.5, NA, 7])

data.dropna()
```

## Exercise:
Run and test the above code.

## `notnull()`: returns list of boolean values indicating whether an item is null or not

```
data.notnull()
```

##  `dropna()` With DataFrame objects, things are a bit more complex.  You may want to drop rows or columns that are all NA or only those containing any NAs. `dropna` by default drops any row containing a missing value.

```
data = pd.DataFrame([[1., 6.5, 3.], [1., NA, NA],
                     [NA, NA, NA], [NA, 6.5, 3.]])

data.dropna()

data.dropna(axis=1)

data.isnull().sum()

data.isnull().sum().sum()

```

## Exercise:
Run and test the above code.

## Passing how='all' will only drop rows that are all NA:

```
data.dropna(how='all')

data = pd.DataFrame([[1., 6.5, 3.], [1., NA, NA],
                     [NA, NA, NA], [NA, 6.5, 3.]])

data[3] = NA

data.dropna(axis=1, how='all')
```

## Exercise:
Run and test the above code.

## Suppose you want to keep only rows containing a certain number of observations. You can indicate this with the thresh argument as `dropna(thresh=2)`.


```
df = pd.DataFrame(np.random.randn(7, 3))
df.iloc[:4, 1] = NA
df.iloc[:2, 2] = NA

df.dropna()
df.dropna(thresh=2)
```

## Exercise:
Run and test the above code.

# Filling In Missing Data
### Rather than filtering out missing data (and potentially discarding other data along with it), you may want to fill in the “holes” in any number of ways.
### For most purposes, the `fillna` method is the workhorse function to use. Calling `fillna` with a constant replaces missing values with that value:

```
data = pd.DataFrame([[1., 2, 3.], [4., NA, NA],
                     [NA, NA, NA], [NA, 8, 9]])


data.fillna(0)

```

Calling fillna with a dict, you can use a different fill value for each column:

```
data.fillna({0: 5, 1: 0.5, 2: 0})

data.fillna(data.mean(), inplace = True)

```


## Exercise:
Run and test the above code.

### fillna returns a new object, but you can modify the existing object in-place:

```
_ = df.fillna(0, inplace=True)

df.fillna({1:0.3, 2:0.5}, inplace=True)

```

## Exercise:
Run and test the above code.

### With fillna you can do lots of other things with a little creativity. For example, you might pass the mean or median value of a Series:

```
data = pd.Series([1., NA, 3.5, NA, 7])

data.fillna(data.median())

```

## We can also fill in the missing value using the previous or next value by `ffill` or `bfill`.
```
df = pd.DataFrame(np.random.randn(6, 3))
df.iloc[2:, 1] = NA
df.iloc[4:, 2] = NA

data.fillna(method='bfill')

df.fillna(method='ffill', limit=2)
```

# Data Transformation

## The transform() function:
- The transform() function in pandas is used to perform operations on each element of a series or column in a DataFrame.
- Unlike apply(), transform() returns an object that is the same size as the input.
- This makes it particularly useful for filling missing values.
- The transform() function is often used with groupby objects to perform operations within groups while maintaining the original DataFrame shape.
```
df = pd.DataFrame({'A': ['foo', 'bar', 'foo', 'bar', 'foo', 'bar', ],
                 'B': [1, np.nan, 2, 4, np.nan, 2]})
B_imputed = df.groupby('A')['B'].transform(lambda x: x.fillna(x.mean()))
```

## Exercise:
Run and test the above code.

## Removing Duplicates


```
data = pd.DataFrame({'k1': ['one', 'two'] * 3 + ['two'],
                     'k2': [1, 1, 2, 3, 3, 4, 4]})

```

## The DataFrame method `duplicated()` returns a boolean Series indicating whether each row is a duplicate (has been observed in a previous row) or not:
```
data.duplicated()

```

### drop_duplicates returns a DataFrame where the duplicated array is removed:
```
data.drop_duplicates()
```

### drop_duplicates can take a column as a parameter.

```
data['v1'] = range(7)

data.drop_duplicates(['k1', 'k2'])

```

### duplicated and drop_duplicates by default keep the first observed value combination. Passing keep='last' will return the last one:

```
data.drop_duplicates(['k1', 'k2'], keep='last')

data.drop_duplicates(keep='last')

```



## Transforming Data Using a Function or Mapping
### For many datasets, you may wish to perform some transformation based on the values in an array, Series, or column in a DataFrame. Consider the following data collected about various kinds of meat:

```
data = pd.DataFrame({'food': ['bacon', 'pulled pork', 'bacon',
                              'Pastrami', 'corned beef', 'Bacon',
                              'pastrami', 'honey ham', 'nova lox'],
                     'ounces': [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})

```

### Suppose we wanted to add a column indicating the type of animal that each food came from.
### Step 1: Write down a mapping of each distinct meat type to the kind of animal:

```
meat_to_animal = {
  'bacon': 'pig',
  'pulled pork': 'pig',
  'pastrami': 'cow',
  'corned beef': 'cow',
  'honey ham': 'pig',
  'nova lox': 'salmon'
}
```

### Step 2: Normalize the text if the map only uses normalized lower-case strings.

```
data['food'].str.lower()

lowercased = data['food'].str.lower()
```

### Step 3: Apply the map method on a Series by accepting a function or dict-like object containing a mapping

```
data['animal'] = data['food'].str.lower().map(meat_to_animal)
```

## We could also have passed a function that does all the work:
```
data['food'].map(lambda x: meat_to_animal[x.lower()])
```

# Replacing Values
Filling in missing data with the fillna method is a special case of more general value
replacement. As you’ve already seen, map can be used to modify a subset of values in
an object.
## `replace` provides a simpler and more flexible way to do so. Let’s consider this Series:

```
data = pd.Series([1., -999., 2., -999., -1000., 3.])
```

The -999 values might be sentinel values for missing data. To replace these with NA
values that pandas understands, we can use replace, producing a new Series (unless
you pass inplace=True):

```
data1 = data.replace([-999, -1000], np.nan)

data1.fillna(data1.mean())
```

## Exercise:
Run and test the above code.

If you want to replace multiple values at once, you instead pass a list and then the
substitute value:

```
data.replace([-999, -1000], np.nan)
```

To use a different replacement for each value, pass a list of substitutes:

```
data.replace([-999, -1000], [np.nan, 0])
```

The argument passed can also be a dict:

```
data.replace({-999: np.nan, -1000: 0})
```

## Renaming Axis Indexes
### Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects.
### We can also modify the axes in-place without creating a new data structure. Here’s a simple example:

```
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=['Ohio', 'Colorado', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
```

### We can modify the index of the DataFrame in-place:

```
transform = lambda x: x[:4].upper()
data.index = data.index.map(transform)
```

### If we want to create a transformed version of a dataset without modifying the original, a useful method is rename:

```
data.rename(index=str.title, columns=str.upper)

data.rename(columns={"one":'column1'})
```

### Notably, rename can be used in conjunction with a dict-like object providing new values for a subset of the axis labels:

```
data.rename(index={'OHIO': 'INDIANA'},
            columns={'three': 'peekaboo'})
```

### rename saves you from the chore of copying the DataFrame manually and assigning to its index and columns attributes. Should you wish to modify a dataset in-place, pass inplace=True:

```
data.rename(index={'OHIO': 'INDIANA'}, inplace=True)
```